# Data Exploration

This notebook explores the processed features dataset for the Explainable AI-Assisted Portfolio Optimization project.

It generates:
- Closing price chart
- Daily return distribution
- Correlation heatmap
- Missing values summary
- Feature statistics table

Figures are saved into `results/figures/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline

# Set up output directory
figures_dir = Path("../results/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

print(f"Figures output directory: {figures_dir.resolve()}")

In [ ]:
# Load the features dataset
df = pd.read_csv("../data/processed/features.csv", parse_dates=["Date"])
print(f"Dataset shape: {df.shape} (rows x columns)")
print(f"Tickers present: {df['Ticker'].unique().tolist()}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Number of unique dates: {df['Date'].nunique()}")

## 1. Closing Price Chart

In [ ]:
# Closing price chart across all tickers over time
plt.figure()
for ticker, group in df.groupby("Ticker"):
    plt.plot(group["Date"], group["Close"], label=ticker, linewidth=1.5)
plt.title("Closing Price Over Time")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / "closing_prices.png")
plt.show()
print("Saved: closing_prices.png")

## 2. Daily Return Distribution

In [ ]:
# Distribution of daily returns
returns = df["Daily_Return"].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(returns, bins=100, alpha=0.7, color="steelblue", edgecolor="white")
axes[0].set_title("Daily Return Distribution (All Tickers)")
axes[0].set_xlabel("Daily Return")
axes[0].set_ylabel("Frequency")
axes[0].grid(True, alpha=0.3)

import scipy.stats as stats
stats.probplot(returns, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot of Daily Returns")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / "daily_return_distribution.png")
plt.show()
print("Saved: daily_return_distribution.png")

## 3. Correlation Heatmap

In [ ]:
# Correlation heatmap of numeric features
feature_columns = [
    "Daily_Return",
    "EMA_20",
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "MACD_Histogram",
    "Momentum_10",
    "Rolling_Volatility_20",
    "Target_Return",
]

corr = df[feature_columns].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Correlation Heatmap of Features")
plt.tight_layout()
plt.savefig(figures_dir / "correlation_heatmap.png")
plt.show()
print("Saved: correlation_heatmap.png")

## 4. Missing Values Summary

In [ ]:
# Missing values summary
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing Percentage (%)": missing_pct.round(2),
})

print("Missing Values Summary:")
print(missing_summary)

missing_summary.to_csv(figures_dir / "missing_values_summary.csv")
print("Saved: missing_values_summary.csv")

## 5. Feature Statistics Table

In [ ]:
# Feature statistics table
stats_df = df[feature_columns].describe().T
stats_df["skew"] = df[feature_columns].skew()
stats_df["kurtosis"] = df[feature_columns].kurt()

print("Feature Statistics Table:")
print(stats_df.round(4))

stats_df.round(6).to_csv(figures_dir / "feature_statistics.csv")
print("\nSaved: feature_statistics.csv")

## Summary

The exploration above gives an overview of the dataset:
- Closing prices across tickers over time
- Distribution of daily returns (with a Q-Q plot for normality check)
- Correlation structure between engineered features and the target
- Missing values (expected at the beginning of each ticker's series due to rolling/EMA calculations)
- Descriptive statistics for all engineered features

All figures are saved in `../../results/figures/`.